# Ghost Carbon — Claimed Green, Metered Gray

**Standalone Kaggle notebook.** Meters train+infer CO₂ (CodeCarbon or labeled PROXY) for RF, LR, tiny MLP vs narrative “negligible/green” claim.

VI: Không phụ thuộc project khác. `pip` cell đầu tiên.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# optional:
# %pip install -q codecarbon torch

In [ ]:
import json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
PROXY_CPU_WATTS, PROXY_GCO2_PER_KWH, CLAIM_KG = 65.0, 475.0, 0.0

def proxy_kgco2e(duration_s):
    kwh = PROXY_CPU_WATTS * duration_s / 3600.0 / 1000.0
    return dict(energy_kwh=kwh, emissions_kg=kwh * PROXY_GCO2_PER_KWH / 1000.0, meter='PROXY_labeled')

def try_tracker():
    try:
        from codecarbon import EmissionsTracker
        return EmissionsTracker(project_name='ghost-carbon', output_dir=str(RESULTS),
                                log_level='error', save_to_file=False, measure_power_secs=1)
    except Exception as e:
        print('CodeCarbon unavailable → PROXY:', e); return None

def measure(fn, label):
    tr = try_tracker(); t0 = time.perf_counter()
    if tr is not None:
        try:
            tr.start(); fn(); em = tr.stop(); dur = time.perf_counter() - t0
            energy = getattr(tr, '_total_energy', None)
            kwh = float(energy.kWh) if energy is not None else float('nan')
            return dict(phase_model=label, duration_s=dur, energy_kwh=kwh,
                        emissions_kg=float(em) if em is not None else float('nan'), meter='codecarbon')
        except Exception as e:
            print('fallback', label, e)
            try: tr.stop()
            except Exception: pass
    fn(); dur = time.perf_counter() - t0
    row = proxy_kgco2e(dur); row.update(phase_model=label, duration_s=dur); return row


In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler(); Xtr, Xte = sc.fit_transform(X_train), sc.transform(X_test)
rows = []
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
rows.append(measure(lambda: rf.fit(X_train, y_train), 'RF_train'))
rows.append(measure(lambda: [rf.predict(X_test) for _ in range(50)], 'RF_infer'))
lr = LogisticRegression(max_iter=500, random_state=42)
rows.append(measure(lambda: lr.fit(Xtr, y_train), 'LR_train'))
rows.append(measure(lambda: [lr.predict(Xte) for _ in range(50)], 'LR_infer'))
mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=200, random_state=42)
rows.append(measure(lambda: mlp.fit(Xtr, y_train), 'MLP_train'))
rows.append(measure(lambda: [mlp.predict(Xte) for _ in range(50)], 'MLP_infer'))
df = pd.DataFrame(rows); df['claim_kg'] = CLAIM_KG; df['ghost_gap_kg'] = df['emissions_kg'] - df['claim_kg']
df.to_csv(RESULTS/'emissions.csv', index=False)
acc = dict(RF=float(rf.score(X_test,y_test)), LR=float(lr.score(Xte,y_test)), MLP=float(mlp.score(Xte,y_test)))
print(acc); display(df)


In [ ]:
families = ['RF','LR','MLP']
metered = [df[df.phase_model.str.startswith(f)].emissions_kg.sum() for f in families]
fig, ax = plt.subplots(figsize=(6,4)); x = np.arange(3)
ax.bar(x-0.2, [CLAIM_KG]*3, 0.4, label='Narrative claim')
ax.bar(x+0.2, metered, 0.4, label='Metered/proxy kgCO2e')
ax.set_xticks(x); ax.set_xticklabels(families); ax.set_ylabel('kgCO2e'); ax.legend()
ax.set_title('Ghost Carbon: claim vs metered'); fig.tight_layout()
fig.savefig(RESULTS/'claim_vs_metered.png', dpi=120); plt.show()
